# COSMoS Observable LOINC Check

**Question.** Do the observable axes derived from the COSMoS package agree with LOINC's own axes for the codes the package pins — and how complete is the COSMoS glucose family against LOINC's?

**Method.** For every LOINC code pinned on an in-scope DSS (`--LOINC`), compare the derivation's coordinates (system, scale, method) against the code's LOINC parts (System, Scale, Method) from Jozef Aerts' XML4Pharma LOINC web services. For glucose, compare the full LOINC similar-test family against the seven derived COSMoS observables.

**Cache-first.** All service responses live in `../cache/loinc_service_cache.json` (fetched 2026-08-24, LOINC v2.82). The notebook runs entirely from cache; codes missing from the cache are fetched live from the service and the cache file is updated. The cached responses are reduced to the axis fields — nothing is invented, and cache provenance is carried into the report.

**Companions.** `COSMoS_Observable_Derivation.ipynb` (produces the input), `docs/Glucose_Siblings_BC_DSS_Proposal.html` (the worked case the glucose sheet informs).

## Inputs

| File | Role |
|---|---|
| `../reports/COSMoS_Observable_Derivation.xlsx` | Derived coordinates and observables |
| `../cache/loinc_service_cache.json` | LOINC axis data per pinned code + glucose similar-test family |

## Output

`../reports/COSMoS_Observable_LOINC_Check.xlsx`

| Sheet | Content |
|---|---|
| README | Provenance, mapping rules, summary counts |
| Axis_Compare | One row per DSS × pinned LOINC code: derived vs LOINC per axis, with verdicts |
| Mismatches | The rows where at least one axis disagrees |
| Glucose_Family | The LOINC glucose family (union of similar-tests) classified against the 7 COSMoS observables |

## 1. Setup

In [1]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('COSMoS OBSERVABLE LOINC CHECK')
print(f'Run: {datetime.now():%Y-%m-%d %H:%M}')

BASE_DIR = Path('..')                       # cosmos-bc-dss/
DERIV_FILE = BASE_DIR / 'reports' / 'COSMoS_Observable_Derivation.xlsx'
CACHE_FILE = BASE_DIR / 'cache' / 'loinc_service_cache.json'
REPORT_FILE = BASE_DIR / 'reports' / 'COSMoS_Observable_LOINC_Check.xlsx'
SERVICE = 'http://www.xml4pharmaserver.com:8080/LOINCService/rest/'

assert DERIV_FILE.exists(), f'Derivation report not found: {DERIV_FILE}'
assert CACHE_FILE.exists(), f'LOINC cache not found: {CACHE_FILE}'

coord = pd.read_excel(DERIV_FILE, 'DSS_Coordinates')
obs = pd.read_excel(DERIV_FILE, 'Observables')
cache = json.loads(CACHE_FILE.read_text())
core = cache['LOINCCoreProperties']
sim = cache['LOINCSimilarTest_glucose_family']
print(f"Coordinates: {len(coord)} DSS   cached core properties: {len(core)}   glucose family: {len(sim)}")
print(f"Cache provenance: LOINC {cache['provenance']['loinc_version']}, fetched {cache['provenance']['fetched']}")

COSMoS OBSERVABLE LOINC CHECK
Run: 2026-08-29 11:02


Coordinates: 1247 DSS   cached core properties: 200   glucose family: 99
Cache provenance: LOINC 2.82, fetched 2026-08-24


## 2. Cache-first fetch

Any pinned code not in the cache is fetched live and the cache file is rewritten. On a machine without
network access this cell is a no-op as long as the cache is complete.

In [2]:
pinned = sorted({c for v in coord['loinc_codes'].dropna() for c in str(v).split(';') if c})
missing = [c for c in pinned if c not in core]
print(f'Pinned LOINC codes: {len(pinned)}   missing from cache: {len(missing)}')

if missing:
    import urllib.request
    for c in missing:
        req = urllib.request.Request(SERVICE + 'LOINCCoreProperties/' + c, headers={'Accept': 'application/json'})
        with urllib.request.urlopen(req, timeout=30) as resp:
            body = json.loads(resp.read().decode('utf-8'))
        t = body['XML4PharmaServerWebServiceResponse']['Response']['LOINCTest']
        core[c] = {k: t.get(k) for k in ('Component', 'Property', 'TimeAspect', 'System', 'ScaleType', 'Method', 'Class', 'ExampleUCUMUnits')}
        print(f'  fetched {c}: {core[c]["Component"]} / {core[c]["System"]} / {core[c]["ScaleType"]}')
    CACHE_FILE.write_text(json.dumps(cache, indent=1, ensure_ascii=False))
    print('Cache updated.')

Pinned LOINC codes: 180   missing from cache: 0


## 3. Mapping rules — LOINC parts vs derived coordinates

Explicit, inspectable tables. A LOINC part maps to a *set* of admissible COSMoS terms; agreement means the
derived value is in the set. Where the derived system is `OPEN` (the DSS offers a specimen choice), LOINC's
system is checked against the offered value list instead — and reported as the pin the package *implies*.

| LOINC scale | Admissible derived scale | Note |
|---|---|---|
| Qn | Quantitative | |
| SemiQn | Quantitative, Coded | semi-quantitative sits on the seam by nature |
| Ord, Nom | Coded | ordinal vs nominal is the resolution the package lacks |
| `-` | (none) | axis-less generic codes |

Where LOINC says Ord/Nom but the derived scale is `Text` **and** the DSS binds no value set on `--ORRES`
(`orres_value_binding = none`), the verdict is `PACKAGE_UNBOUND`, not `DISAGREE`: the package makes no
claim a value set could contradict — the finding is the missing value set itself, not a conflict with LOINC.

The system map covers only the LOINC systems that actually occur among the pinned codes; anything outside
it is reported `UNMAPPED`, never guessed.

In [3]:
SCALE_MAP = {
    'Qn': {'Quantitative'},
    'SemiQn': {'Quantitative', 'Coded'},
    'Ord': {'Coded'},
    'Nom': {'Coded'},
}

SYSTEM_MAP = {
    'Ser': {'SERUM'},
    'Plas': {'PLASMA'},
    'Ser/Plas': {'SERUM', 'PLASMA', 'SERUM OR PLASMA'},
    'Bld': {'BLOOD', 'WHOLE BLOOD'},
    'BldA': {'ARTERIAL BLOOD', 'BLOOD'},
    'PPP': {'PLASMA', 'PLATELET POOR PLASMA'},
    'Urine': {'URINE'},
    'Urine sed': {'URINE', 'URINE SEDIMENT'},
    'Interstitial fluid': {'INTERSTITIAL FLUID'},
    'RBC': {'ERYTHROCYTES', 'RED BLOOD CELLS', 'BLOOD'},
    'Bone mar': {'BONE MARROW'},
    'Stool^Patient.gastrointestinal': {'FECES', 'STOOL'},
    'Air': set(),
    'XXX': set(),
    'BldA+Inhl gas': set(),
}


def scale_verdict(loinc_scale, derived_scale, orres_value_binding):
    if loinc_scale not in SCALE_MAP:
        return 'UNMAPPED'
    if derived_scale in SCALE_MAP[loinc_scale]:
        return 'AGREE'
    if loinc_scale in ('Ord', 'Nom') and derived_scale == 'Text' and orres_value_binding == 'none':
        return 'PACKAGE_UNBOUND'
    return 'DISAGREE'


def system_verdict(loinc_system, derived_system, spec_value_list):
    if loinc_system not in SYSTEM_MAP:
        return 'UNMAPPED'
    admissible = SYSTEM_MAP[loinc_system]
    if not admissible:
        return 'NOT_COMPARABLE'
    if derived_system in admissible:
        return 'AGREE'
    if derived_system == 'OPEN':
        offered = {t.strip() for t in str(spec_value_list).split(';')} if pd.notna(spec_value_list) else set()
        return 'OPEN_COMPATIBLE' if admissible & offered else 'OPEN_INCOMPATIBLE'
    if derived_system == 'NONE':
        return 'NO_DERIVED_SYSTEM'
    return 'DISAGREE'


def method_verdict(loinc_method, derived_method):
    if loinc_method and derived_method not in ('OPEN', 'NONE', 'UNBOUND'):
        return 'BOTH_PINNED'
    if loinc_method:
        return 'LOINC_ONLY'      # LOINC says method matters; the DSS does not pin one
    if derived_method not in ('OPEN', 'NONE', 'UNBOUND'):
        return 'COSMOS_ONLY'     # the DSS pins a method LOINC considers non-identifying
    return 'NEITHER'

## 4. Axis comparison — every DSS × pinned code

The DSS grain (not the observable grain) is compared, because the pin sits on the DSS. The specimen
value list is re-read from the graph only via the derivation report's columns — no second source.

In [4]:
rows = []
for r in coord.itertuples(index=False):
    if pd.isna(r.loinc_codes):
        continue
    for c in str(r.loinc_codes).split(';'):
        if c not in core:
            rows.append({'ds_id': r.ds_id, 'loinc_code': c, 'error': 'not in cache'})
            continue
        t = core[c]
        rows.append({
            'ds_id': r.ds_id, 'bc_id': r.bc_id, 'bc_short_name': r.bc_short_name, 'domain': r.domain,
            'loinc_code': c, 'loinc_component': t['Component'], 'loinc_property': t['Property'],
            'loinc_time': t['TimeAspect'], 'loinc_system': t['System'], 'loinc_scale': t['ScaleType'],
            'loinc_method': t['Method'], 'loinc_class': t['Class'], 'loinc_units': t['ExampleUCUMUnits'],
            'derived_system': r.system, 'derived_scale': r.scale, 'derived_method': r.method,
            'orres_value_binding': r.orres_value_binding,
        })
comp = pd.DataFrame(rows)
assert comp['loinc_code'].notna().all()

In [5]:
# The specimen value list is not carried per-row in the derivation report's coordinate sheet,
# so OPEN systems are checked against the *unit* of comparability only when the graph offers it.
# Re-derive the offered specimen lists from the graph file for the OPEN rows.
GRAPH_FILE = BASE_DIR / '..' / 'cosmos-graph' / 'interim' / 'COSMoS_Graph.xlsx'
gvar = pd.read_excel(GRAPH_FILE, 'Variables')
gvar['suffix'] = gvar['variable_name'].str[2:]
spec_lists = (gvar[gvar['suffix'] == 'SPEC'].set_index('ds_id')['value_list'])

comp['spec_value_list'] = comp['ds_id'].map(spec_lists)
comp['system_verdict'] = [system_verdict(t.loinc_system, t.derived_system, t.spec_value_list) for t in comp.itertuples(index=False)]
comp['scale_verdict'] = [scale_verdict(t.loinc_scale, t.derived_scale, t.orres_value_binding) for t in comp.itertuples(index=False)]
comp['method_verdict'] = [method_verdict(t.loinc_method, t.derived_method) for t in comp.itertuples(index=False)]

print(f'DSS x LOINC pairs compared: {len(comp)}')
print()
print('System :', comp['system_verdict'].value_counts().to_dict())
print('Scale  :', comp['scale_verdict'].value_counts().to_dict())
print('Method :', comp['method_verdict'].value_counts().to_dict())

mismatch = comp[comp['system_verdict'].isin(['DISAGREE', 'OPEN_INCOMPATIBLE', 'UNMAPPED'])
                | comp['scale_verdict'].isin(['DISAGREE', 'UNMAPPED', 'PACKAGE_UNBOUND'])]
print(f'\nRows with a system or scale finding: {len(mismatch)}')
print(f"  of which PACKAGE_UNBOUND (package binds no value set - not a LOINC disagreement): "
      f"{int((mismatch['scale_verdict'] == 'PACKAGE_UNBOUND').sum())}")
if len(mismatch):
    print(mismatch[['ds_id', 'loinc_code', 'loinc_component', 'loinc_system', 'derived_system',
                    'system_verdict', 'loinc_scale', 'derived_scale', 'scale_verdict']].to_string(index=False))

DSS x LOINC pairs compared: 183

System : {'AGREE': 177, 'NOT_COMPARABLE': 3, 'DISAGREE': 3}
Scale  : {'AGREE': 152, 'PACKAGE_UNBOUND': 30, 'DISAGREE': 1}
Method : {'NEITHER': 159, 'LOINC_ONLY': 18, 'BOTH_PINNED': 6}

Rows with a system or scale finding: 34
  of which PACKAGE_UNBOUND (package binds no value set - not a LOINC disagreement): 30
          ds_id loinc_code                               loinc_component                   loinc_system  derived_system system_verdict loinc_scale derived_scale   scale_verdict
    ALBURINPRES     1753-3                                       Albumin                          Urine           URINE          AGREE         Ord          Text PACKAGE_UNBOUND
ANGLOBDRRBCPRES     1006-6 Direct antiglobulin test.IgG specific reagent                            RBC    ERYTHROCYTES          AGREE         Nom          Text PACKAGE_UNBOUND
       ANISOBLD    38892-6                                  Anisocytosis                            Bld           BLOOD     

## 5. The unit-dimension pairs — property axis as recording variant

Where one DSS pins two codes, LOINC's Property axis should differ by mass-vs-substance concentration
(MCnc vs SCnc) — the unit dimension this work classifies as *recording*, not identity. Verify.

In [6]:
multi = comp.groupby('ds_id').filter(lambda g: len(g) > 1)
pairs = (multi.groupby('ds_id')
         .agg(bc_short_name=('bc_short_name', 'first'),
              codes=('loinc_code', ';'.join),
              properties=('loinc_property', lambda s: ';'.join(sorted(set(s)))),
              systems=('loinc_system', lambda s: ';'.join(sorted(set(s)))),
              scales=('loinc_scale', lambda s: ';'.join(sorted(set(s)))))
         .reset_index())
pairs['same_system'] = ~pairs['systems'].str.contains(';')
pairs['same_scale'] = ~pairs['scales'].str.contains(';')
pairs['property_only_pair'] = pairs['same_system'] & pairs['same_scale'] & pairs['properties'].str.contains(';')
print(f'DSS with more than one pinned code: {len(pairs)}')
print(f'  differing ONLY on the property axis (unit dimension - recording variants): {int(pairs.property_only_pair.sum())}')
print(f'  crossing system or scale inside one DSS: {int((~pairs.same_system | ~pairs.same_scale).sum())}')
odd = pairs[~pairs['same_system'] | ~pairs['same_scale']]
if len(odd):
    print(odd[['ds_id', 'bc_short_name', 'codes', 'systems', 'scales']].to_string(index=False))

DSS with more than one pinned code: 42
  differing ONLY on the property axis (unit dimension - recording variants): 39
  crossing system or scale inside one DSS: 0


## 6. Glucose family completeness

The cached `LOINCSimilarTest` union over the ten pinned glucose codes (99 codes) classified against the
seven derived COSMoS observables. Classification is mechanical, from the LOINC parts:

- **pinned in COSMoS** — the code is on a glucose DSS today.
- **recording variant of a covered observable** — same system group and scale as a pinned code; differs
  only on property (unit dimension), method grain LOINC carries, or a documentation code (`{Measurement}`).
- **time-aspect variant** — a timed collection (24H, 12H, …) or mean over a period; COSMoS has no
  collection-duration axis for glucose.
- **system gap** — a specimen COSMoS has no glucose observable for (CSF, capillary blood, dialysis
  fluid, …).
- **non-clinical / other** — water, non-biological fluid, TPN.

In [7]:
GLUCOSE_PINNED = {c for v in coord.loc[coord['bc_id'] == 'C105585', 'loinc_codes'].dropna() for c in str(v).split(';')}
COVERED_SYSTEM_GROUPS = {'Ser/Plas': 'blood-derived', 'Ser': 'blood-derived', 'Plas': 'blood-derived',
                         'Bld': 'blood-derived', 'Interstitial fluid': 'plasma-equivalent', 'Urine': 'urine'}
GAP_SYSTEMS_NONCLINICAL = {'Water', 'Flu.nonbiological', 'TPN'}

fam = []
for c, t in sim.items():
    sysname, scale, timeaspect, prop, method = t['System'], t['ScaleType'], t['TimeAspect'], t['Property'], t['Method']
    if c in GLUCOSE_PINNED:
        cls = 'pinned in COSMoS'
    elif prop == '{Measurement}':
        cls = 'generic documentation code'
    elif timeaspect not in ('Pt', None):
        cls = 'time-aspect variant (no COSMoS axis)'
    elif sysname in GAP_SYSTEMS_NONCLINICAL:
        cls = 'non-clinical / other'
    elif sysname in COVERED_SYSTEM_GROUPS:
        cls = 'recording variant of covered observable'
    else:
        cls = 'system gap'
    fam.append({'loinc_code': c, 'component': t['Component'], 'property': prop, 'time': timeaspect,
                'system': sysname, 'scale': scale, 'method': method, 'classification': cls,
                'covered_group': COVERED_SYSTEM_GROUPS.get(sysname)})
fam = pd.DataFrame(fam).sort_values(['classification', 'system', 'loinc_code'])
print(f'LOINC glucose family: {len(fam)} codes')
print()
print(fam['classification'].value_counts().to_string())
print()
gaps = fam[fam['classification'] == 'system gap']
print('System gaps (specimens LOINC has, COSMoS glucose does not):')
print(gaps.groupby('system').size().sort_values(ascending=False).to_string())

LOINC glucose family: 99 codes

classification
system gap                                 41
time-aspect variant (no COSMoS axis)       20
recording variant of covered observable    13
generic documentation code                 11
pinned in COSMoS                           10
non-clinical / other                        4

System gaps (specimens LOINC has, COSMoS glucose does not):
system
BldC            4
Stool           4
BldA            3
BldV            3
Dial fld prt    2
Ser/Plas/Bld    2
Vitr fld        2
BldMV           2
Body fld        2
CSF             2
Dial fld        2
Synv fld        2
Pericard fld    2
Periton fld     2
Plr fld         2
Amnio fld       1
Gast fld        1
Bld.dot         1
BldCo           1
XXX             1


## 7. Report

In [8]:
YELLOW, YELLOW_DATA = 'FFD700', 'FFFCE8'
BLUE, BLUE_DATA = '2E5C8A', 'EAF1F8'      # blue = LOINC service content (external source)
GREY, GREY_DATA = '7F7F7F', 'F2F2F2'      # grey = derived / verdicts
THIN = Side(style='thin', color='999999')
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

LOINC_COLS = {'loinc_code', 'loinc_component', 'loinc_property', 'loinc_time', 'loinc_system', 'loinc_scale',
              'loinc_method', 'loinc_class', 'loinc_units', 'component', 'property', 'time', 'system',
              'scale', 'method', 'codes', 'properties', 'systems', 'scales'}
DERIVED_COLS = {'derived_system', 'derived_scale', 'derived_method', 'system_verdict', 'scale_verdict',
                'method_verdict', 'classification', 'covered_group', 'same_system', 'same_scale',
                'property_only_pair', 'spec_value_list', 'orres_value_binding'}


def write_sheet(wb, name, df, wide=()):
    ws = wb.create_sheet(name)
    ws.append(list(df.columns))
    for c, col in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=c)
        fill = GREY if col in DERIVED_COLS else (BLUE if col in LOINC_COLS else YELLOW)
        cell.fill = PatternFill('solid', fgColor=fill)
        cell.font = Font(bold=True, color='FFFFFF')
        cell.border = BORDER
        cell.alignment = Alignment(wrap_text=True, vertical='top')
    for r in df.itertuples(index=False):
        ws.append([None if (isinstance(v, float) and pd.isna(v)) else v for v in r])
    for rowcells in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in rowcells:
            col = df.columns[cell.column - 1]
            cell.fill = PatternFill('solid', fgColor=GREY_DATA if col in DERIVED_COLS else (BLUE_DATA if col in LOINC_COLS else YELLOW_DATA))
            cell.border = BORDER
            cell.alignment = Alignment(wrap_text=True, vertical='top')
    for c, col in enumerate(df.columns, 1):
        width = max([len(str(col))] + [len(str(v)) for v in df[col].head(300)])
        ws.column_dimensions[get_column_letter(c)].width = min(width + 2, 70 if col in wide else 34)
    ws.freeze_panes = 'A2'


n_pin_dss = int((coord['loinc_n'] > 0).sum())
pin_err = comp[comp['system_verdict'] == 'DISAGREE']
dup_pins = (comp.groupby('loinc_code').filter(lambda g: g['bc_id'].nunique() > 1)
            .groupby('loinc_code').agg(ds=('ds_id', lambda s: ';'.join(sorted(set(s))))))
scale_real = comp[comp['scale_verdict'] == 'DISAGREE']
n_unbound = int((comp['scale_verdict'] == 'PACKAGE_UNBOUND').sum())
n_time = int((fam['classification'] == 'time-aspect variant (no COSMoS axis)').sum())

sys_counts = comp['system_verdict'].value_counts().to_dict()
sc_counts = comp['scale_verdict'].value_counts().to_dict()
me_counts = comp['method_verdict'].value_counts().to_dict()
readme_lines = [
    'COSMoS Observable LOINC Check',
    '',
    f'Generated: {datetime.now():%Y-%m-%d %H:%M}',
    f"LOINC source: XML4Pharma LOINC web services ({SERVICE}), LOINC {cache['provenance']['loinc_version']}, cache fetched {cache['provenance']['fetched']}",
    'Inputs: reports/COSMoS_Observable_Derivation.xlsx + cache/loinc_service_cache.json',
    'Notebook: cosmos-bc-dss/notebooks/COSMoS_Observable_LOINC_Check.ipynb',
    '',
    'QUESTION',
    "Do the observable axes derived from the package agree with LOINC's own axes for the pinned codes, and how complete is the COSMoS glucose family against LOINC's?",
    '',
    'MAPPING RULES',
    'Scale: Qn->Quantitative; SemiQn->{Quantitative,Coded}; Ord/Nom->Coded (ordinal-vs-nominal is resolution the package lacks); "-" not comparable.',
    'PACKAGE_UNBOUND: LOINC Ord/Nom vs derived Text where the DSS binds no value set on --ORRES - no package claim to disagree with; the gap is the missing value set.',
    'System: explicit LOINC-system -> COSMoS-term sets (see notebook section 3); systems outside the map report UNMAPPED, never guessed.',
    'OPEN derived system: LOINC system checked against the specimen value list the DSS offers -> OPEN_COMPATIBLE means LOINC names the pin the package implies.',
    'Method: BOTH_PINNED / LOINC_ONLY (LOINC says method matters, DSS does not pin) / COSMOS_ONLY / NEITHER.',
    '',
    'SUMMARY',
    f'DSS x pinned-LOINC pairs compared: {len(comp)}',
    f'Coverage: {n_pin_dss} of {len(coord)} in-scope DSS carry a LOINC pin - agreement validates the derivation rules on the pinned subset, not the corpus.',
    'System verdicts: ' + ', '.join(f'{k}: {v}' for k, v in sys_counts.items()),
    'Scale verdicts: ' + ', '.join(f'{k}: {v}' for k, v in sc_counts.items()),
    'Method verdicts: ' + ', '.join(f'{k}: {v}' for k, v in me_counts.items()),
    f'Rows with a system or scale finding: {len(mismatch)} (PACKAGE_UNBOUND {n_unbound}; real scale disagreements {len(scale_real)}; system disagreements {len(pin_err)})',
    f"Multi-code DSS: {len(pairs)}, of which property-axis-only (unit-dimension recording variants): {int(pairs.property_only_pair.sum())}",
    f"Glucose family: {len(fam)} LOINC codes; " + ', '.join(f'{k}: {v}' for k, v in fam['classification'].value_counts().items()),
    '',
    'FINDINGS',
    'Suspected COSMoS pin errors (system DISAGREE, graph-verified):',
] + [
    f'  {r.ds_id}: pins {r.loinc_code} ({r.loinc_component}; LOINC system {r.loinc_system}) on derived system {r.derived_system}'
    for r in pin_err.itertuples(index=False)
] + [
    'Duplicate pin across BCs (one LOINC observable, two COSMoS BCs - the reverse of the glucose case): '
    + '; '.join(f'{code} on {row.ds}' for code, row in dup_pins.iterrows()),
    f'Scale: {n_unbound} of {n_unbound + len(scale_real)} non-agreeing scale rows are PACKAGE_UNBOUND - the missing-value-set finding holds corpus-wide; real disagreements: '
    + ', '.join(f'{r.ds_id} ({r.loinc_scale} vs {r.derived_scale})' for r in scale_real.itertuples(index=False)),
    f'Glucose family: {n_time} timed-collection variants in LOINC with no COSMoS collection-duration axis.',
    '',
    'COLOR CONVENTION',
    'Yellow FFD700 = COSMoS package identifiers. Blue 2E5C8A = LOINC service content (external source). Grey 7F7F7F = derived values and verdicts.',
]

wb = Workbook()
ws = wb.active
ws.title = 'README'
ws.column_dimensions['A'].width = 90
ws['A1'] = readme_lines[0]
ws['A1'].fill = PatternFill('solid', fgColor='595959')
ws['A1'].font = Font(bold=True, color='FFFFFF')
for i, line in enumerate(readme_lines[1:], start=2):
    ws.cell(row=i, column=1, value=line).alignment = Alignment(wrap_text=True, vertical='top')

order = ['ds_id', 'bc_id', 'bc_short_name', 'domain', 'loinc_code', 'loinc_component', 'loinc_property',
         'loinc_time', 'loinc_system', 'loinc_scale', 'loinc_method', 'loinc_class', 'loinc_units',
         'derived_system', 'derived_scale', 'derived_method', 'orres_value_binding', 'spec_value_list',
         'system_verdict', 'scale_verdict', 'method_verdict']
write_sheet(wb, 'Axis_Compare', comp[order], wide=('spec_value_list', 'loinc_component'))
write_sheet(wb, 'Mismatches', mismatch[order], wide=('spec_value_list', 'loinc_component'))
write_sheet(wb, 'Multi_Code_DSS', pairs)
write_sheet(wb, 'Glucose_Family', fam)
wb.save(REPORT_FILE)
print(f'Written: {REPORT_FILE}')

Written: ../reports/COSMoS_Observable_LOINC_Check.xlsx
